# 4B. Analisis Spasial: Hotspot & Cold Spot (RQ2, RQ4)

Metode (lihat `docs/analysis_plan.md` bagian 6, Draft):
- Variabel: rata-rata pickup per jam valid per zona per bulan (jam DST hilang tidak dihitung).
- Spatial weights: KNN k=8 pada centroid zona (EPSG:2263); sensitivitas k=6 dan k=10.
- Global Moran's I dan Getis-Ord Gi\* (999 permutasi, seed 42), uji dua sisi.
- Koreksi FDR Benjamini-Hochberg, α=0,05. Hotspot: q < 0,05 dan Zs > 0. Cold spot: q < 0,05 dan Zs < 0.
- Stabilitas antarbulan: Jaccard Index.

Koordinat adalah **centroid zona**, bukan GPS asli. Input: `datasets/interim/2026/demand_zone_hour.parquet`.

## 0. Import & Konfigurasi

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
import folium
from libpysal.weights import KNN
from esda.moran import Moran
from esda.getisord import G_Local
from statsmodels.stats.multitest import multipletests

warnings.filterwarnings("ignore", category=FutureWarning)

SEED = 42
PERMS = 999
ALPHA = 0.05
K_MAIN = 8
K_SENS = (6, 10)

DATA = "../datasets/interim/2026/demand_zone_hour.parquet"
SHP = "../datasets/zone/taxi_zones/taxi_zones.shp"
OUT_TAB = "../out/output_eda"
OUT_HTML = "../out/html"
os.makedirs(OUT_TAB, exist_ok=True)
os.makedirs(OUT_HTML, exist_ok=True)
BULAN = {1: "Jan", 2: "Feb", 3: "Mar", 4: "Apr", 5: "Mei", 6: "Jun"}

## 1. Muat Data & Geometri

In [ ]:
grid = pd.read_parquet(DATA, columns=["hour", "zone_id", "pickup_count", "is_dst_missing"])
grid["month"] = grid["hour"].dt.month
print("Baris grid:", len(grid), "| zona:", grid["zone_id"].nunique(), "| NaN pickup:", grid["pickup_count"].isna().sum())

zones = gpd.read_file(SHP)
zones = zones.dissolve(by="LocationID", as_index=False)   # satu zona = satu record
zones = zones.rename(columns={"LocationID": "zone_id", "zone": "zone_name"})
assert zones.crs.to_epsg() == 2263
zones = zones[zones["zone_id"].isin(grid["zone_id"].unique())].sort_values("zone_id").reset_index(drop=True)
print("Zona analisis spasial:", len(zones), "(264 dan 265 tidak ada di shapefile)")

## 2. Variabel: Rata-rata Pickup per Jam Valid per Zona per Bulan
`mean` melewatkan NaN, jadi otomatis dinormalisasi jumlah jam valid (Feb 28 hari, Mar ada jam DST hilang).

In [ ]:
mean_zm = (grid.groupby(["zone_id", "month"])["pickup_count"].mean()
           .unstack("month").reindex(zones["zone_id"]))
mean_zm.columns = [BULAN[m] for m in mean_zm.columns]
mean_all = grid.groupby("zone_id")["pickup_count"].mean().reindex(zones["zone_id"])
print(mean_zm.describe().round(2))
assert not mean_zm.isna().any().any()

## 3. Spatial Weights (KNN pada centroid, EPSG:2263)

In [ ]:
cent = zones.geometry.centroid
pts = np.column_stack([cent.x, cent.y])

def make_w(k):
    w = KNN.from_array(pts, k=k)
    w.transform = "R"
    return w

w8 = make_w(K_MAIN)
print(f"KNN k={K_MAIN}: island={len(w8.islands)}, komponen terhubung={w8.n_components}")
for k in K_SENS:
    wk = make_w(k)
    print(f"KNN k={k}: island={len(wk.islands)}, komponen terhubung={wk.n_components}")

## 4. Global Moran's I per Bulan

In [ ]:
rows = []
for b in mean_zm.columns:
    np.random.seed(SEED)
    mi = Moran(mean_zm[b].values, w8, permutations=PERMS)
    rows.append({"bulan": b, "moran_I": mi.I, "E[I]": mi.EI, "z_sim": mi.z_sim, "p_sim": mi.p_sim})
np.random.seed(SEED)
mi_all = Moran(mean_all.values, w8, permutations=PERMS)
rows.append({"bulan": "Total", "moran_I": mi_all.I, "E[I]": mi_all.EI, "z_sim": mi_all.z_sim, "p_sim": mi_all.p_sim})
moran_tbl = pd.DataFrame(rows)
moran_tbl.to_csv(f"{OUT_TAB}/04b_moran_global.csv", index=False)
moran_tbl.round(4)

## 5. Getis-Ord Gi\* + Koreksi FDR
`alternative="two-sided"` supaya p-value layak dikoreksi FDR. FDR dihitung **per bulan** pada 263 uji lokal.

In [ ]:
def run_gistar(y, w, seed=SEED):
    g = G_Local(np.asarray(y, dtype=float), w, transform="R", star=True,
                permutations=PERMS, seed=seed, alternative="two-sided")
    q = multipletests(g.p_sim, alpha=ALPHA, method="fdr_bh")[1]
    cls = np.where((q < ALPHA) & (g.Zs > 0), "Hotspot",
          np.where((q < ALPHA) & (g.Zs < 0), "Cold spot", "Tidak signifikan"))
    return pd.DataFrame({"Zs": g.Zs, "p_sim": g.p_sim, "q_fdr": q, "kelas": cls})

res = {}
for b in mean_zm.columns:
    r = run_gistar(mean_zm[b].values, w8)
    r.insert(0, "zone_id", zones["zone_id"].values)
    r.insert(1, "zone_name", zones["zone_name"].values)
    r.insert(2, "bulan", b)
    res[b] = r
gi = pd.concat(res.values(), ignore_index=True)
gi.to_csv(f"{OUT_TAB}/04b_gistar_bulanan.csv", index=False)

ringkas = gi.pivot_table(index="bulan", columns="kelas", values="zone_id", aggfunc="count", fill_value=0)
ringkas = ringkas.reindex(list(BULAN.values()))
ringkas

## 6. Stabilitas Hotspot Antarbulan (Jaccard Index)

In [ ]:
def jaccard(a, b):
    a, b = set(a), set(b)
    return len(a & b) / len(a | b) if (a | b) else np.nan

bulan = list(BULAN.values())
def jaccard_matrix(label):
    sets = {b: res[b].loc[res[b]["kelas"] == label, "zone_id"] for b in bulan}
    m = pd.DataFrame(index=bulan, columns=bulan, dtype=float)
    for i in bulan:
        for j in bulan:
            m.loc[i, j] = jaccard(sets[i], sets[j])
    return m

J_hot, J_cold = jaccard_matrix("Hotspot"), jaccard_matrix("Cold spot")
J_hot.to_csv(f"{OUT_TAB}/04b_jaccard_hotspot.csv")
J_cold.to_csv(f"{OUT_TAB}/04b_jaccard_coldspot.csv")

fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))
for a, m, t in zip(ax, (J_hot, J_cold), ("Hotspot", "Cold spot")):
    sns.heatmap(m, annot=True, fmt=".2f", vmin=0, vmax=1, cmap="YlOrRd", ax=a, cbar=False)
    a.set_title(f"Jaccard {t} antarbulan")
plt.tight_layout()
plt.savefig(f"{OUT_TAB}/04b_jaccard_heatmap.png", dpi=150)
plt.show()

iu = np.triu_indices(len(bulan), k=1)
print("Rata-rata Jaccard hotspot :", round(float(np.nanmean(J_hot.values[iu])), 3))
print("Rata-rata Jaccard cold spot:", round(float(np.nanmean(J_cold.values[iu])), 3))

## 7. Hotspot yang Konsisten & Daftar Prioritas (masukan RQ4)
Zona hotspot di semua 6 bulan = kandidat zona prioritas.

In [ ]:
cnt = (gi.assign(h=(gi["kelas"] == "Hotspot").astype(int), c=(gi["kelas"] == "Cold spot").astype(int))
         .groupby(["zone_id", "zone_name"])[["h", "c"]].sum().reset_index()
         .rename(columns={"h": "bulan_hotspot", "c": "bulan_cold"}))
cnt = cnt.merge(mean_all.rename("rata2_pickup_jam").reset_index(), on="zone_id")
prioritas = cnt[cnt["bulan_hotspot"] == 6].sort_values("rata2_pickup_jam", ascending=False)
cnt.to_csv(f"{OUT_TAB}/04b_hotspot_konsistensi.csv", index=False)
print("Hotspot 6/6 bulan:", len(prioritas), "| Cold spot 6/6 bulan:", int((cnt["bulan_cold"] == 6).sum()))
prioritas.head(15)

## 8. Sensitivitas k (k=6, 8, 10) pada Rata-rata Seluruh Periode

In [ ]:
sens = {}
for k in (K_MAIN, *K_SENS):
    r = run_gistar(mean_all.values, make_w(k))
    sens[k] = set(zones["zone_id"][r["kelas"] == "Hotspot"])
    print(f"k={k}: hotspot={len(sens[k])}")
print("Jaccard k8 vs k6 :", round(jaccard(sens[8], sens[6]), 3))
print("Jaccard k8 vs k10:", round(jaccard(sens[8], sens[10]), 3))

## 9. Peta Hotspot / Cold Spot (HTML, label: centroid zona)

In [ ]:
z4326 = zones.to_crs(4326)
WARNA = {"Hotspot": "#d7301f", "Cold spot": "#2b6cb0", "Tidak signifikan": "#d9d9d9"}
m = folium.Map(location=[40.72, -73.95], zoom_start=10, tiles="cartodbpositron")
for b in bulan:
    fg = folium.FeatureGroup(name=f"Gi* {b}", show=(b == bulan[0]))
    d = z4326[["zone_id", "zone_name", "geometry"]].merge(res[b][["zone_id", "Zs", "q_fdr", "kelas"]], on="zone_id")
    folium.GeoJson(
        d.to_json(),
        style_function=lambda f: {"fillColor": WARNA[f["properties"]["kelas"]], "color": "#555",
                                  "weight": 0.5, "fillOpacity": 0.75},
        tooltip=folium.GeoJsonTooltip(fields=["zone_name", "kelas", "Zs", "q_fdr"],
                                      aliases=["Zona", "Kelas", "Z-score", "q FDR"], localize=True),
    ).add_to(fg)
    fg.add_to(m)
folium.LayerControl(collapsed=False).add_to(m)
m.get_root().html.add_child(folium.Element(
    "<div style='position:fixed;bottom:20px;left:20px;z-index:9999;background:white;padding:8px;font-size:12px'>"
    "Gi* (KNN k=8, FDR-BH 0,05). Variabel: rata-rata pickup/jam. Lokasi = centroid zona.</div>"))
m.save(f"{OUT_HTML}/04b_hotspot_bulanan.html")
print("Tersimpan:", f"{OUT_HTML}/04b_hotspot_bulanan.html")

## 10. Jawaban RQ2 & Catatan
Isi setelah notebook dijalankan pada data final:
- Apakah ada autokorelasi spasial (Moran's I per bulan)?
- Zona hotspot / cold spot utama dan seberapa stabil (rata-rata Jaccard)?
- Apakah hasil sensitif terhadap k?
- Zona prioritas untuk RQ4 (hotspot 6/6 bulan).

Catat keputusan yang berubah di `docs/analysis_plan.md` bagian 14.